# Predator-Prey Agent-Based Simulation
## Capstone Project Notebook

The predator-prey relationship is one of the most fundamental ecological interactions. In classical ecology, it is described by the **Lotka-Volterra equations** — a pair of coupled ordinary differential equations that produce characteristic oscillations: prey numbers rise, predators follow with a lag, predators overexploit the prey, predators decline, prey recover, and the cycle repeats.

However, the Lotka-Volterra model assumes a **well-mixed population** where every predator has equal access to every prey individual. In reality, ecological interactions are profoundly **spatial**: predators must search for prey, prey can hide in refugia, and local extinctions can occur even as the global population persists. These spatial effects lead to **emergent phenomena** — patterns at the population level that arise from individual-level rules but cannot be predicted from the ODE model alone.

**Agent-based models (ABMs)** address these limitations by simulating each individual organism as an autonomous agent with its own position, energy, and behavioural rules. The agents move through space, interact locally, and their collective dynamics produce population-level patterns. ABMs naturally capture:

- **Spatial heterogeneity** — predators and prey are not uniformly distributed
- **Stochasticity** — random encounters, births, and deaths create variability between runs
- **Emergent clustering** — prey may form herds, predators may concentrate in prey-rich areas
- **Local extinction and recolonisation** — patches can go extinct while the metapopulation persists
- **Individual variation** — agents can differ in speed, energy reserves, or behaviour

### In this notebook you will:
1. Review the classical **Lotka-Volterra** ODE model as a baseline
2. Build an **agent-based predator-prey** simulation step by step
3. Implement movement, hunting, reproduction, and energy mechanics
4. Compare ABM population oscillations with ODE predictions
5. Explore stochastic variability and parameter sensitivity
6. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for numerical operations, Matplotlib for plotting, and Python's `random` module for stochastic agent decisions. We define colour constants for prey (green) and predators (red) that will be used consistently in all plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 12})
C_PREY = '#2ecc71'; C_PRED = '#e74c3c'
print('Imports loaded.')

---
## 1 · Lotka-Volterra Baseline

Before building the ABM, we establish a baseline using the classical **Lotka-Volterra** ODE model:

$$\frac{dx}{dt} = \alpha x - \beta x y, \qquad \frac{dy}{dt} = \delta x y - \gamma y$$

where $x$ = prey population, $y$ = predator population, and:
- $\alpha$ — prey birth rate (exponential growth in the absence of predators)
- $\beta$ — predation rate (rate at which prey are consumed per predator)
- $\delta$ — predator efficiency (how much energy predators gain from eating prey)
- $\gamma$ — predator death rate (starvation in the absence of prey)

This deterministic model produces perfectly periodic oscillations: prey grow, predators follow with a lag, predators overexploit prey, predators starve, prey recover. These oscillations are **structurally unstable** — any perturbation changes the orbit permanently. The ABM we build next will show how stochasticity and spatial structure alter this idealised picture.

In [ ]:
def lotka_volterra(x, y, alpha, beta, delta, gamma):
    dx = alpha * x - beta * x * y
    dy = delta * x * y - gamma * y
    return dx, dy

def simulate_lv(x0, y0, alpha, beta, delta, gamma, T, dt=0.01):
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    x, y = np.zeros(steps), np.zeros(steps)
    x[0], y[0] = x0, y0
    for k in range(1, steps):
        dx, dy = lotka_volterra(x[k-1], y[k-1], alpha, beta, delta, gamma)
        x[k] = max(0, x[k-1] + dt * dx)
        y[k] = max(0, y[k-1] + dt * dy)
    return t, x, y

t_lv, x_lv, y_lv = simulate_lv(40, 9, alpha=1.0, beta=0.1, delta=0.075, gamma=1.5, T=30)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_lv, x_lv, color=C_PREY, lw=2, label='Prey')
ax.plot(t_lv, y_lv, color=C_PRED, lw=2, label='Predators')
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title('Lotka-Volterra ODE Baseline', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

---
## 2 · Agent Design

Each agent is an individual organism with its own state:
- **Position** $(x, y)$ on a 2D grid — determines who it can interact with
- **Type**: prey or predator — determines behaviour rules
- **Energy**: a scalar that increases when the agent feeds and decreases each step (movement cost). If energy reaches zero, the agent dies.

The `Agent` class below encapsulates this state. The `move` method implements a random walk: each step the agent moves one cell in a random cardinal direction (N/S/E/W), which costs 1 energy. Agents are confined to the grid boundaries.

This energy-based design creates a natural life cycle: agents must feed to survive and accumulate enough energy to reproduce. Predators that fail to catch prey will eventually starve; prey that graze efficiently can build up energy for reproduction.

In [ ]:
class Agent:
    def __init__(self, x, y, agent_type, energy, rng):
        self.x = x
        self.y = y
        self.agent_type = agent_type  # 'prey' or 'predator'
        self.energy = energy
        self.alive = True
        self.rng = rng
    
    def move(self, area):
        direction = self.rng.randint(1, 4)
        if direction == 1 and self.y < area: self.y += 1
        elif direction == 2 and self.y > 0: self.y -= 1
        elif direction == 3 and self.x > 0: self.x -= 1
        elif direction == 4 and self.x < area: self.x += 1
        self.energy -= 1
        if self.energy <= 0:
            self.alive = False

# Test
rng = random.Random(42)
a = Agent(10, 10, 'prey', 20, rng)
a.move(40)
assert a.energy == 19
assert a.alive
print('Agent test passed.')

---
## 3 · Simulation Core

The simulation loop orchestrates all agent interactions each time step. The order of operations matters — it determines which events take priority:

1. **Movement** — all agents take one random step (costs 1 energy). Agents that run out of energy die.
2. **Grazing** — prey gain energy passively (representing grass consumption). This is the energy input that drives the entire food web.
3. **Predation** — each predator checks if any prey shares its cell. If so, it eats one prey (prey dies, predator gains energy). Only one meal per predator per step — this limits predator efficiency and prevents instantaneous prey collapse.
4. **Reproduction** — agents whose energy exceeds a threshold split: the parent keeps half its energy and a new offspring is created at the same location with the other half.
5. **Cleanup** — dead agents are removed from the simulation.

The function returns time series of prey and predator counts. Compare these oscillations with the smooth Lotka-Volterra curves above — the ABM produces **irregular, noisy oscillations** that can include extinction events.

In [ ]:
def simulate_predator_prey(area=50, n_prey=100, n_pred=20,
                              prey_energy=20, pred_energy=40,
                              prey_gain=4, pred_gain=20,
                              prey_repro=15, pred_repro=30,
                              steps=300, seed=42):
    rng = random.Random(seed)
    agents = []
    for _ in range(n_prey):
        agents.append(Agent(rng.randint(0, area), rng.randint(0, area),
                           'prey', prey_energy, rng))
    for _ in range(n_pred):
        agents.append(Agent(rng.randint(0, area), rng.randint(0, area),
                           'predator', pred_energy, rng))
    
    prey_hist, pred_hist = [], []
    
    for step in range(steps):
        # Count
        prey_count = sum(1 for a in agents if a.agent_type == 'prey' and a.alive)
        pred_count = sum(1 for a in agents if a.agent_type == 'predator' and a.alive)
        prey_hist.append(prey_count)
        pred_hist.append(pred_count)
        
        if prey_count == 0 and pred_count == 0:
            break
        
        # Move all
        for a in agents:
            if a.alive: a.move(area)
        
        # Prey gain energy (grazing)
        for a in agents:
            if a.alive and a.agent_type == 'prey':
                a.energy += prey_gain
        
        # Predation: predator eats one prey on same cell
        alive_prey = [a for a in agents if a.alive and a.agent_type == 'prey']
        for pred in agents:
            if not pred.alive or pred.agent_type != 'predator':
                continue
            for prey in alive_prey:
                if prey.alive and prey.x == pred.x and prey.y == pred.y:
                    prey.alive = False
                    pred.energy += pred_gain
                    break  # one meal per step
        
        # Reproduction
        new_agents = []
        for a in agents:
            if not a.alive: continue
            threshold = prey_repro if a.agent_type == 'prey' else pred_repro
            if a.energy >= threshold:
                a.energy //= 2
                child = Agent(a.x, a.y, a.agent_type, a.energy, rng)
                new_agents.append(child)
        agents.extend(new_agents)
        
        # Remove dead
        agents = [a for a in agents if a.alive]
    
    return np.array(prey_hist), np.array(pred_hist)

prey_h, pred_h = simulate_predator_prey(seed=42)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(prey_h, color=C_PREY, lw=2, label='Prey')
ax.plot(pred_h, color=C_PRED, lw=2, label='Predators')
ax.set_xlabel('Time step'); ax.set_ylabel('Population')
ax.set_title('Agent-Based Predator-Prey Dynamics', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()
print(f'Final: {prey_h[-1]} prey, {pred_h[-1]} predators')

---
## 4 · Stochastic Variability

Because every movement step, every predation encounter, and every reproduction event involves randomness, no two ABM runs are alike — even with identical parameters. Below we overlay 10 independent runs to illustrate this variability. Some runs show sustained oscillations, others may see predators go extinct early (leaving prey to grow unchecked) or prey collapse (causing predator starvation).

This stochastic variability is not a weakness — it reflects the **real ecological uncertainty** that conservation managers face. It also means that summary statistics (mean, variance, extinction probability) over many runs are more informative than any single trajectory.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for s in range(1, 11):
    ph, _ = simulate_predator_prey(seed=s)
    ax.plot(ph, color=C_PREY, lw=0.8, alpha=0.5)
    _, prh = simulate_predator_prey(seed=s)
    ax.plot(prh, color=C_PRED, lw=0.8, alpha=0.5)
ax.set_xlabel('Time step'); ax.set_ylabel('Population')
ax.set_title('10 ABM Runs — Stochastic Variability', fontweight='bold')
plt.tight_layout(); plt.show()
print('Each run produces different oscillation patterns and timing.')

---
## 5 · Parameter Sensitivity

Three key parameters control the balance between predators and prey:

- **Initial predator count** (left) — more predators means faster initial prey depletion. Too many predators can drive prey to extinction before the ecosystem stabilises.
- **Prey energy gain** (centre) — higher grazing efficiency means prey reproduce faster, supporting a larger predator population. This is analogous to the prey growth rate $\alpha$ in the Lotka-Volterra model.
- **Grid size** (right) — a larger grid at the same population count means lower density. Lower density reduces encounter rates between predators and prey, weakening the coupling and potentially allowing both populations to persist longer in a loosely connected state.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Vary initial predator count
for n_pred, c in zip([5, 15, 30], ['#3498db', '#e67e22', '#e74c3c']):
    ph, prh = simulate_predator_prey(n_pred=n_pred, seed=42)
    axes[0].plot(ph, color=c, lw=1.5, label=f'{n_pred} pred')
axes[0].set_title('Vary Initial Predators'); axes[0].legend(fontsize=9)

# Vary prey gain
for pg, c in zip([2, 4, 8], ['#3498db', '#e67e22', '#e74c3c']):
    ph, prh = simulate_predator_prey(prey_gain=pg, seed=42)
    axes[1].plot(ph, color=c, lw=1.5, label=f'gain={pg}')
axes[1].set_title('Vary Prey Energy Gain'); axes[1].legend(fontsize=9)

# Vary grid size (density)
for ar, c in zip([30, 50, 80], ['#3498db', '#e67e22', '#e74c3c']):
    ph, prh = simulate_predator_prey(area=ar, seed=42)
    axes[2].plot(ph, color=c, lw=1.5, label=f'area={ar}')
axes[2].set_title('Vary Grid Size (Density)'); axes[2].legend(fontsize=9)

for ax in axes:
    ax.set_xlabel('Step'); ax.set_ylabel('Prey count')
plt.tight_layout(); plt.show()

---
## 6 · Your Tasks

Implement the core ABM (done above) and add **at least two** features:

### Task A: Multi-Level Interactions
Introduce a food chain (predators → prey → vegetation). Vegetation grows on the grid, prey must eat it, predators eat prey.

### Task B: Terrain Factors
Different terrain types: water (barrier), forest (prey hide, reduced predation probability), open grassland (fast movement).

### Task C: Seasonal Changes
Vary prey energy gain or reproduction threshold cyclically to simulate seasons. Show how seasons affect oscillation patterns.

### Task D: Energy & Consumption Limits
Add maximum energy, digestion time, competition for prey on same cell.

### Discussion points
- Compare ABM oscillations to Lotka-Volterra ODE predictions
- Show emergent behaviours (extinction events, stable coexistence)
- Demonstrate with animations or GIFs
- Discuss stochasticity and its ecological implications

In [ ]:
# ============================================================
# PLACEHOLDER: Implement your extensions below
# ============================================================

# Task A: Multi-Level Interactions
# TODO

# Task B: Terrain Factors
# TODO

# Task C: Seasonal Changes
# TODO

# Task D: Energy & Consumption Limits
# TODO

---
## Recommended Reading & Journal Club

### Foundational References

**1. Lotka, A. J. (1925)** *Elements of Physical Biology.* Williams & Wilkins.
→ Original formulation of the predator-prey equations.

**2. Volterra, V. (1926)** *Fluctuations in the abundance of a species considered mathematically.* Nature, 118, 558–560.

---

### Journal Club Papers

**3. Wilensky, U. & Rand, W. (2015)** *An Introduction to Agent-Based Modeling.* MIT Press.
→ Excellent textbook with predator-prey as a running example.

**4. Grimm, V. et al. (2006)** *A standard protocol for describing individual-based and agent-based models.* Ecological Modelling, 198(1–2), 115–126. [DOI](https://doi.org/10.1016/j.ecolmodel.2006.04.023)
→ The ODD protocol for documenting ABMs — follow this for your report.

**5. Railsback, S. F. & Grimm, V. (2019)** *Agent-Based and Individual-Based Modeling.* 2nd ed. Princeton University Press.
→ Comprehensive guide to ABM in ecology.